# Extractive Summarization — TextRank

Metode TextRank: menilai kalimat lewat relasi antar-kalimat (graf + PageRank). Kalimat yang mirip dengan banyak kalimat lain dianggap paling penting. Input: `data_final_preprocessing.csv` (sudah dipreprocessing).

**Struktur:** Load → Stopword → Fungsi → ROUGE → BERTScore → Contoh → Simpan.

---
## Setup

In [6]:
!pip install rouge-score bert-score -q

In [7]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from tqdm import tqdm
tqdm.pandas()

---
## 1. Load Dataset

In [8]:
df = pd.read_csv('/kaggle/input/datasets/nazhifberlian/nlp-wikipedia-summarization/data_final_preprocessing.csv',
                 encoding='utf-8-sig')

print(f'Dataset dimuat: {len(df)} baris, {len(df.columns)} kolom')
print(f'Kolom: {list(df.columns)}')
print(f'\nDistribusi kategori:')
print(df['category'].value_counts().to_string())

Dataset dimuat: 12644 baris, 9 kolom
Kolom: ['global_id', 'id', 'title', 'category', 'article_text', 'body_word_count', 'lead_paragraph', 'lead_word_count', 'sentences']

Distribusi kategori:
category
sejarah     1976
arts        1962
artis       1947
kuliner     1947
tech        1716
biografi    1679
sains       1417


---
## 2. Stopword Removal
* Stopword diabaikan saat membangun matriks kemiripan antar-kalimat (di dalam `TfidfVectorizer`).
* Stopword (dipakai di matriks kemiripan), langkah 3 graf + PageRank untuk cari kalimat paling sentral.

In [9]:
STOPWORDS_ID = set("""
yang di ke dari dan atau pada untuk dengan adalah ini itu dalam tidak akan
sebagai juga oleh karena sudah dapat telah agar bisa saat para suatu setelah
hingga maupun sebuah merupakan namun yaitu serta secara hanya masih lebih tersebut
""".split())

print(f'Jumlah stopword: {len(STOPWORDS_ID)}')

Jumlah stopword: 40


---
## 3. Fungsi Summarization

In [10]:
def split_sentences(text):
    sentences = sent_tokenize(str(text))
    return [s.strip() for s in sentences if len(s.split()) >= 4]

def adaptive_n(text):
    n = len(str(text).split())
    if n < 500:  return 2
    if n < 1500: return 3
    return 4

def summarize_textrank(text, n_sentences=3):
    sentences = split_sentences(text)
    if len(sentences) <= n_sentences:
        return ' '.join(sentences)
    vectorizer = TfidfVectorizer(stop_words=list(STOPWORDS_ID), lowercase=True)
    tfidf_matrix = vectorizer.fit_transform(sentences)
    sim_matrix = cosine_similarity(tfidf_matrix)
    np.fill_diagonal(sim_matrix, 0)
    graph = nx.from_numpy_array(sim_matrix)
    scores = nx.pagerank(graph, max_iter=100)
    ranked = sorted(scores, key=scores.get, reverse=True)
    top_idx = sorted(ranked[:n_sentences])
    return ' '.join(sentences[i] for i in top_idx)

print('Fungsi TextRank siap.')

Fungsi TextRank siap.


### Generate ringkasan untuk seluruh dataset

In [11]:
print('Generating ringkasan TextRank...')
start = time.time()

df['summary_textrank'] = df['article_text'].progress_apply(
    lambda x: summarize_textrank(x, n_sentences=adaptive_n(x)))

time_textrank = time.time() - start
print(f'Selesai dalam {time_textrank:.1f} detik ({time_textrank/len(df):.3f} detik/artikel)')

Generating ringkasan TextRank...


100%|██████████| 12644/12644 [01:02<00:00, 201.52it/s]

Selesai dalam 62.7 detik (0.005 detik/artikel)


---
## 4. Evaluasi ROUGE

In [12]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

r1, r2, rl = [], [], []
for pred, ref in zip(df['summary_textrank'], df['lead_paragraph']):
    s = scorer.score(str(ref), str(pred))
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rl.append(s['rougeL'].fmeasure)

rouge_textrank = (np.mean(r1), np.mean(r2), np.mean(rl))

print('Evaluasi ROUGE — TextRank')
print(f'{"Metrik":<10} | {"Skor":>8}')
print('-' * 22)
print(f'{"ROUGE-1":<10} | {rouge_textrank[0]:>8.4f}')
print(f'{"ROUGE-2":<10} | {rouge_textrank[1]:>8.4f}')
print(f'{"ROUGE-L":<10} | {rouge_textrank[2]:>8.4f}')

Evaluasi ROUGE — TextRank
Metrik     |     Skor
----------------------
ROUGE-1    |   0.2024
ROUGE-2    |   0.0500
ROUGE-L    |   0.1269


### ROUGE-L per Kategori

In [13]:
rows = []
for kat in sorted(df['category'].unique()):
    sub = df[df['category'] == kat]
    scores = [scorer.score(str(ref), str(pred))['rougeL'].fmeasure
              for pred, ref in zip(sub['summary_textrank'], sub['lead_paragraph'])]
    rows.append({'kategori': kat, 'jumlah': len(sub), 'rougeL': np.mean(scores)})

hasil_kategori = pd.DataFrame(rows).sort_values('rougeL', ascending=False)
print('ROUGE-L per kategori (TextRank):')
print(hasil_kategori.round(4).to_string(index=False))

ROUGE-L per kategori (TextRank):
kategori  jumlah  rougeL
 sejarah    1976  0.1378
    tech    1716  0.1330
 kuliner    1947  0.1324
   sains    1417  0.1245
    arts    1962  0.1210
   artis    1947  0.1200
biografi    1679  0.1183


---
## 5. Evaluasi BERTScore

In [14]:
SAMPLE_SIZE_BS = 500
sample_bs = df.sample(n=min(SAMPLE_SIZE_BS, len(df)), random_state=42).reset_index(drop=True)

cands = sample_bs['summary_textrank'].astype(str).tolist()
refs  = sample_bs['lead_paragraph'].astype(str).tolist()

P, R, F1 = bertscore(cands, refs, lang='id', verbose=False)
bertscore_textrank = (P.mean().item(), R.mean().item(), F1.mean().item())

print(f'Evaluasi BERTScore — TextRank (sampel N={len(sample_bs)})')
print(f'{"Metrik":<12} | {"Skor":>8}')
print('-' * 24)
print(f'{"Precision":<12} | {bertscore_textrank[0]:>8.4f}')
print(f'{"Recall":<12} | {bertscore_textrank[1]:>8.4f}')
print(f'{"F1":<12} | {bertscore_textrank[2]:>8.4f}')

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluasi BERTScore — TextRank (sampel N=500)
Metrik       |     Skor
------------------------
Precision    |   0.6830
Recall       |   0.6565
F1           |   0.6687


---
## 6. Contoh Ringkasan Hasil

In [17]:
N_CONTOH = 3
sample_idx = df.sample(N_CONTOH, random_state=42).index

for i, idx in enumerate(sample_idx):
    row = df.loc[idx]
    print('=' * 70)
    print(f'Artikel {i+1}: [{row["category"]}] {row["title"]}')
    print('-' * 20)
    print(f'Ground Truth (lead_paragraph):')
    print(f'{str(row["lead_paragraph"])[:200]}...')
    print('-' * 20)
    print(f'Ringkasan TextRank:')
    print(f'{str(row["summary_textrank"])[:200]}')
    print()

Artikel 1: [sains] Atom Bakhidrogen
--------------------
Ground Truth (lead_paragraph):
suatu ion lirhidrogen bahasa inggris hydrogenlike ioncode en is deprecated  disebut pula ion mirip hidrogen merupakan inti atom yang memiliki satu elektron dan karenanya isoelektronik dengan hidrogen....
--------------------
Ringkasan TextRank:
dalam penyelesaian persamaan schrdinger, yang bersifat nonrelativistik, orbital atom bakhidrogen merupakan eigenfungsi dari operator momentum sudut satuelektron l dan komponen znya lz. berbagai nilai 

Artikel 2: [kuliner] Jeruk Bergamot
--------------------
Ground Truth (lead_paragraph):
jeruk bergamot adalah buah jeruk wangi dengan warna kuning atau hijau mirip dengan jeruk nipis, tergantung pada kematangannya. penelitian genetika tentang asal muasal leluhur kultivar jeruk yang masih...
--------------------
Ringkasan TextRank:
jeruk bergamot tidak ada hubungannya dengan tanaman herbal yang dikenal sebagai bergamot, bergamot liar, bergamot mint, atau bergami

---
## 7. Simpan Hasil

In [16]:
output_cols = ['global_id', 'title', 'category', 'summary_textrank',
               'lead_paragraph', 'body_word_count']
hasil = df[[c for c in output_cols if c in df.columns]]

hasil.to_csv('hasil_summary_textrank.csv', index=False, encoding='utf-8-sig')
print(f'Tersimpan: hasil_summary_textrank.csv ({len(hasil)} baris)')
print(f'Kolom: {list(hasil.columns)}')

Tersimpan: hasil_summary_textrank.csv (12644 baris)
Kolom: ['global_id', 'title', 'category', 'summary_textrank', 'lead_paragraph', 'body_word_count']
